# Metacognitive Calibration

**Track:** Metacognition
**Construct:** Confidence-accuracy correspondence

Tests whether a model's stated confidence in its answers correlates with its actual accuracy. Well-calibrated models should be right ~80% of the time when they say they are 80% confident.

## Cognitive Science Background

**Calibration** measures the correspondence between stated confidence and actual accuracy (Fischhoff, Slovic & Lichtenstein, 1977). Systematic overconfidence — the **Dunning-Kruger effect** — is a hallmark of poor metacognition (Kruger & Dunning, 1999). This benchmark adapts the metacognitive monitoring framework (Nelson & Narens, 1990) by measuring retrospective confidence across diverse question types.

**Human baseline ECE:** 0.10–0.20

## Methodology

Questions spanning multiple domains (trivia, math, reasoning, estimation) are presented. The model answers each question and provides a confidence rating (0–100%). Responses are binned by confidence level, and accuracy per bin is computed to calculate Expected Calibration Error (ECE) and Brier Skill Score (BSS).

## Scoring

$$\text{Score} = 0.50 \\text{times} \text{extreme\_accuracy}^{1.5} + 0.25 \\text{times} \text{bss\_normalized} + 0.25 \\text{times} \text{uncertainty\_awareness}$$

Three components:
- **Extreme accuracy (50%):** Accuracy on difficulty ≥ 4 questions, raised to the power of 1.5 to amplify differences between models
- **BSS normalized (25%):** Brier Skill Score mapped from [-1, 1] to [0, 1], rewarding both calibration and resolution
- **Uncertainty awareness (25%):** `1 - mean_confidence_on_hard_incorrect`, rewarding low confidence on hard questions answered incorrectly

Final score is clamped to [0, 1].

| Score | Interpretation |
|:---:|---|
| 0.8–1.0 | Excellent calibration — confidence tracks accuracy precisely across difficulty levels |
| 0.5–0.8 | Good — generally calibrated but overconfident on hard items |
| 0.2–0.5 | Moderate — systematic miscalibration (overconfidence or underconfidence) |
| 0.0–0.2 | Poor — confidence bears little relation to actual accuracy |

### References

Nelson & Narens (1990), Fischhoff et al. (1977), Kruger & Dunning (1999)

In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

In [ ]:
"""
Calibration benchmark question dataset v2.

Redesigned to produce a meaningful spread of accuracy across frontier models
(GPT-4o, Gemini 2.5 Pro, Claude Opus). The v1 set was too easy — models
answered ~99% at confidence=100, collapsing BSS.

Design principles:
- 5 difficulty tiers targeting different accuracy bands
- Emphasis on precise numerical recall, common misconceptions, contested facts,
  tricky logic, cross-domain precision, and commonly confabulated specifics
- Every question has an unambiguous, verifiable correct answer
- No procedurally generated arithmetic (handled by procedural_calibration.py)
- Minimal overlap with well-known quiz/trivia datasets
- accept_patterns provided for answers with multiple valid phrasings

Difficulty tiers:
  1 = easy   (~95%+ accuracy)  — baseline anchor
  2 = medium (~70-85%)         — requires specific knowledge
  3 = hard   (~40-60%)         — obscure facts, precise numbers, misconceptions
  4 = very hard (~15-35%)      — things LLMs commonly get wrong
  5 = extreme  (~5-15%)        — near-impossible without lookup
"""

CALIBRATION_QUESTIONS = [
    # =========================================================================
    # TIER 1: Easy (~15 questions) — baseline anchor, ~95%+ accuracy
    # =========================================================================
    {
        "question": "What is the chemical symbol for gold?",
        "answer": "Au",
        "domain": "chemistry",
        "difficulty": 1,
        "accept_patterns": ["Au"],
    },
    {
        "question": "How many sides does a hexagon have?",
        "answer": "6",
        "domain": "math",
        "difficulty": 1,
        "accept_patterns": ["6", "six"],
    },
    {
        "question": "What planet is known as the Red Planet?",
        "answer": "Mars",
        "domain": "astronomy",
        "difficulty": 1,
        "accept_patterns": ["Mars"],
    },
    {
        "question": "In which year did World War II end?",
        "answer": "1945",
        "domain": "history",
        "difficulty": 1,
        "accept_patterns": ["1945"],
    },
    {
        "question": "What is the capital of Japan?",
        "answer": "Tokyo",
        "domain": "geography",
        "difficulty": 1,
        "accept_patterns": ["Tokyo"],
    },
    {
        "question": "What does DNA stand for?",
        "answer": "deoxyribonucleic acid",
        "domain": "biology",
        "difficulty": 1,
        "accept_patterns": ["deoxyribonucleic acid"],
    },
    {
        "question": "How many hearts does an octopus have?",
        "answer": "3",
        "domain": "biology",
        "difficulty": 1,
        "accept_patterns": ["3", "three"],
    },
    {
        "question": "What is the smallest country in the world by area?",
        "answer": "Vatican City",
        "domain": "geography",
        "difficulty": 1,
        "accept_patterns": ["Vatican", "Vatican City"],
    },
    {
        "question": "In what year was Wikipedia launched?",
        "answer": "2001",
        "domain": "technology",
        "difficulty": 1,
        "accept_patterns": ["2001"],
    },
    {
        "question": "What is the largest organ in the human body?",
        "answer": "skin",
        "domain": "biology",
        "difficulty": 1,
        "accept_patterns": ["skin"],
    },
    {
        "question": "Who wrote the play 'Romeo and Juliet'?",
        "answer": "William Shakespeare",
        "domain": "literature",
        "difficulty": 1,
        "accept_patterns": ["Shakespeare"],
    },
    {
        "question": "What is the boiling point of water in degrees Celsius at standard atmospheric pressure?",
        "answer": "100",
        "domain": "physics",
        "difficulty": 1,
        "accept_patterns": ["100"],
    },
    {
        "question": "How many bones are in the adult human body?",
        "answer": "206",
        "domain": "anatomy",
        "difficulty": 1,
        "accept_patterns": ["206"],
    },
    {
        "question": "What is the speed of sound in air at 20°C, in meters per second?",
        "answer": "343",
        "domain": "physics",
        "difficulty": 1,
        "accept_patterns": ["343"],
    },
    {
        "question": "Which country was the first to grant women the right to vote in national elections?",
        "answer": "New Zealand",
        "domain": "history",
        "difficulty": 1,
        "accept_patterns": ["New Zealand"],
    },

    # =========================================================================
    # TIER 2: Medium (~15 questions) — requires specific knowledge, ~70-85%
    # =========================================================================
    {
        "question": "What is the half-life of Carbon-14, approximately in years?",
        "answer": "5730",
        "domain": "physics",
        "difficulty": 2,
        "accept_patterns": ["5730", "5,730"],
    },
    {
        "question": "In what year was the Treaty of Tordesillas signed, dividing the New World between Spain and Portugal?",
        "answer": "1494",
        "domain": "history",
        "difficulty": 2,
        "accept_patterns": ["1494"],
    },
    {
        "question": "What is the densest naturally occurring element?",
        "answer": "osmium",
        "domain": "chemistry",
        "difficulty": 2,
        "accept_patterns": ["osmium", "Os"],
    },
    {
        "question": "How many time zones does Russia span?",
        "answer": "11",
        "domain": "geography",
        "difficulty": 2,
        "accept_patterns": ["11", "eleven"],
    },
    {
        "question": "What element has the highest melting point?",
        "answer": "tungsten",
        "domain": "chemistry",
        "difficulty": 2,
        "accept_patterns": ["tungsten", "W", "wolfram"],
    },
    {
        "question": "In what year was the first network email sent by Ray Tomlinson?",
        "answer": "1971",
        "domain": "technology",
        "difficulty": 2,
        "accept_patterns": ["1971"],
    },
    {
        "question": "How many US states border the Pacific Ocean?",
        "answer": "5",
        "domain": "geography",
        "difficulty": 2,
        "accept_patterns": ["5", "five"],
    },
    {
        "question": "What is the Mohs hardness of quartz?",
        "answer": "7",
        "domain": "geology",
        "difficulty": 2,
        "accept_patterns": ["7", "seven"],
    },
    {
        "question": "In what year did the Berlin Wall fall?",
        "answer": "1989",
        "domain": "history",
        "difficulty": 2,
        "accept_patterns": ["1989"],
    },
    {
        "question": "How many completed novels did Jane Austen write?",
        "answer": "6",
        "domain": "literature",
        "difficulty": 2,
        "accept_patterns": ["6", "six"],
    },
    {
        "question": "In what year was the Battle of Hastings fought?",
        "answer": "1066",
        "domain": "history",
        "difficulty": 2,
        "accept_patterns": ["1066"],
    },
    {
        "question": "What is the exact height of the Burj Khalifa in meters (to the tip)?",
        "answer": "828",
        "domain": "architecture",
        "difficulty": 2,
        "accept_patterns": ["828", "829.8"],
    },
    {
        "question": "How many recognized countries are in Africa according to the United Nations?",
        "answer": "54",
        "domain": "geography",
        "difficulty": 2,
        "accept_patterns": ["54"],
    },
    {
        "question": "In what year was the first edition of the Encyclopaedia Britannica published?",
        "answer": "1768",
        "domain": "history",
        "difficulty": 2,
        "accept_patterns": ["1768"],
    },
    {
        "question": "What is the standard atmospheric pressure at sea level in pascals?",
        "answer": "101325",
        "domain": "physics",
        "difficulty": 2,
        "accept_patterns": ["101325", "101,325"],
    },

    # =========================================================================
    # TIER 3: Hard (~20 questions) — obscure facts, misconceptions, ~40-60%
    # =========================================================================
    {
        "question": "What percentage of Earth's water is fresh water (not salt water)? Give to one decimal place.",
        "answer": "2.5",
        "domain": "earth science",
        "difficulty": 3,
        "accept_patterns": ["2.5", "3", "2.5%", "3%"],
    },
    {
        "question": "What is the driest continent on Earth by average annual precipitation?",
        "answer": "Antarctica",
        "domain": "geography",
        "difficulty": 3,
        "accept_patterns": ["Antarctica"],
    },
    {
        "question": "Which country has the most islands in the world?",
        "answer": "Sweden",
        "domain": "geography",
        "difficulty": 3,
        "accept_patterns": ["Sweden"],
    },
    {
        "question": "Is the Great Wall of China visible to the naked eye from low Earth orbit?",
        "answer": "No",
        "domain": "science",
        "difficulty": 3,
        "accept_patterns": ["No", "no", "not visible", "cannot"],
    },
    {
        "question": "What is the value of the fine-structure constant (alpha) to 4 significant figures? Express as a decimal.",
        "answer": "0.007297",
        "domain": "physics",
        "difficulty": 3,
        "accept_patterns": ["0.007297", "0.00730", "1/137"],
    },
    {
        "question": "How many U.S. presidents have died while in office (including assassinations)?",
        "answer": "8",
        "domain": "history",
        "difficulty": 3,
        "accept_patterns": ["8", "eight"],
    },
    {
        "question": "Which planet in our solar system currently has the most known moons?",
        "answer": "Saturn",
        "domain": "astronomy",
        "difficulty": 3,
        "accept_patterns": ["Saturn"],
    },
    {
        "question": "In what year was the earliest surviving photograph (by Nicéphore Niépce) taken?",
        "answer": "1826",
        "domain": "history",
        "difficulty": 3,
        "accept_patterns": ["1826", "1827"],
    },
    {
        "question": "What is the exact value of the Planck constant h in J·s, as defined in the 2019 SI? Give all significant digits.",
        "answer": "6.62607015e-34",
        "domain": "physics",
        "difficulty": 3,
        "accept_patterns": ["6.62607015", "6.626 070 15"],
    },
    {
        "question": "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost in dollars?",
        "answer": "0.05",
        "domain": "math",
        "difficulty": 3,
        "accept_patterns": ["0.05", "$0.05", "5 cents", "five cents"],
    },
    {
        "question": "If you have a 4x4x4 cube made of 64 small unit cubes, and you paint the outside, how many unit cubes have exactly two painted faces?",
        "answer": "24",
        "domain": "math",
        "difficulty": 3,
        "accept_patterns": ["24"],
    },
    {
        "question": "In what year were human chromosomes correctly counted as 46 (not 48)?",
        "answer": "1955",
        "domain": "biology",
        "difficulty": 3,
        "accept_patterns": ["1955", "1956"],
    },
    {
        "question": "What is the sum of all integers from 1 to 100?",
        "answer": "5050",
        "domain": "math",
        "difficulty": 3,
        "accept_patterns": ["5050", "5,050"],
    },
    {
        "question": "In the original Monty Hall problem, what is the probability of winning if you switch doors? Express as a fraction.",
        "answer": "2/3",
        "domain": "math",
        "difficulty": 3,
        "accept_patterns": ["2/3", "0.667", "0.66", "66.7%"],
    },
    {
        "question": "How many plays are in the traditional Shakespeare canon (First Folio plus Pericles)?",
        "answer": "37",
        "domain": "literature",
        "difficulty": 3,
        "accept_patterns": ["37"],
    },
    {
        "question": "What is the exact value of the Avogadro constant as defined in the 2019 SI redefinition, in mol⁻¹?",
        "answer": "6.02214076e23",
        "domain": "chemistry",
        "difficulty": 3,
        "accept_patterns": ["6.02214076"],
    },
    {
        "question": "In what year did Anders Celsius propose his temperature scale?",
        "answer": "1742",
        "domain": "history of science",
        "difficulty": 3,
        "accept_patterns": ["1742"],
    },
    {
        "question": "What is the melting point of tungsten in degrees Celsius, rounded to the nearest degree?",
        "answer": "3422",
        "domain": "chemistry",
        "difficulty": 3,
        "accept_patterns": ["3422", "3,422", "3410", "3414"],
    },
    {
        "question": "How many edges does an icosahedron have?",
        "answer": "30",
        "domain": "math",
        "difficulty": 3,
        "accept_patterns": ["30"],
    },
    {
        "question": "What is the exact value of the Boltzmann constant k in J/K as defined in the 2019 SI?",
        "answer": "1.380649e-23",
        "domain": "physics",
        "difficulty": 3,
        "accept_patterns": ["1.380649"],
    },

    # =========================================================================
    # TIER 4: Very Hard (~15 questions) — LLMs commonly get wrong, ~15-35%
    # =========================================================================
    {
        "question": "How many groups of order 8 exist up to isomorphism? (Counting all groups of order 8.)",
        "answer": "5",
        "domain": "abstract algebra",
        "difficulty": 4,
        "accept_patterns": ["5", "five"],
    },
    {
        "question": "In a room of 23 people, what is the probability that at least two share a birthday? Give as a percentage rounded to the nearest whole number.",
        "answer": "50",
        "domain": "probability",
        "difficulty": 4,
        "accept_patterns": ["50", "50%", "51"],
    },
    {
        "question": "What was the population of Liechtenstein in 2024, to the nearest thousand?",
        "answer": "40000",
        "domain": "geography",
        "difficulty": 4,
        "accept_patterns": ["40000", "40,000", "39000", "39,000", "41000", "41,000"],
    },
    {
        "question": "What is the only letter that does not appear in the name of any US state?",
        "answer": "Q",
        "domain": "trivia",
        "difficulty": 4,
        "accept_patterns": ["Q", "q"],
    },
    {
        "question": "How many two-digit prime numbers are there?",
        "answer": "21",
        "domain": "math",
        "difficulty": 4,
        "accept_patterns": ["21"],
    },
    {
        "question": "What is the surface area of a sphere with radius 7, in terms of exact value? Give a numerical answer rounded to 2 decimal places.",
        "answer": "615.75",
        "domain": "math",
        "difficulty": 4,
        "accept_patterns": ["615.75", "615.8", "196*pi", "196π"],
    },
    {
        "question": "In a standard 52-card deck, what is the probability of being dealt a royal flush in 5-card poker? Express as '1 in N' where N is the answer.",
        "answer": "649740",
        "domain": "probability",
        "difficulty": 4,
        "accept_patterns": ["649740", "649,740"],
    },
    {
    "question": "Three people check into a hotel room that costs $30. They each pay $10. The manager realizes the room should only cost $25, so he gives $5 to the bellboy to return. The bellboy keeps $2 and gives back $1 to each guest. Now each guest has paid $9 (total $27), plus the bellboy has $2, totaling $29. Where is the missing dollar?",
        "answer": "There is no missing dollar. The $27 paid includes the $25 for the room plus the $2 the bellboy kept. The $29 figure incorrectly adds cost and tip.",
        "domain": "logic",
        "difficulty": 4,
        "accept_patterns": ["no missing dollar", "there is no missing", "misdirection", "accounting error", "fallacy", "includes"],
    },
    {
        "question": "What is the atomic number of the element Hassium?",
        "answer": "108",
        "domain": "chemistry",
        "difficulty": 4,
        "accept_patterns": ["108"],
    },
    {
        "question": "In what year was the Treaty of Nerchinsk signed between Russia and the Qing Dynasty?",
        "answer": "1689",
        "domain": "history",
        "difficulty": 4,
        "accept_patterns": ["1689"],
    },
    {
        "question": "What is the 10th digit of pi after the decimal point?",
        "answer": "5",
        "domain": "math",
        "difficulty": 4,
        "accept_patterns": ["5"],
    },
    {
        "question": "How many perfect numbers are known to exist as of 2024?",
        "answer": "52",
        "domain": "math",
        "difficulty": 4,
        "accept_patterns": ["52"],
    },
    {
        "question": "If you fold a standard piece of paper (0.1mm thick) in half 42 times, approximately how thick would it be? Answer in kilometers to the nearest thousand.",
        "answer": "440000",
        "domain": "math",
        "difficulty": 4,
        "accept_patterns": ["440000", "440,000", "439804", "439,804"],
    },
    {
        "question": "What is the name of the enzyme that catalyzes the conversion of carbon dioxide and water into glucose during the Calvin cycle in photosynthesis?",
        "answer": "RuBisCO",
        "domain": "biochemistry",
        "difficulty": 4,
        "accept_patterns": ["RuBisCO", "rubisco", "ribulose-1,5-bisphosphate carboxylase", "ribulose bisphosphate carboxylase"],
    },
    {
        "question": "If you have 12 identical-looking balls, one of which is either heavier or lighter than the rest, what is the minimum number of weighings on a balance scale needed to identify the odd ball and determine if it is heavier or lighter?",
        "answer": "3",
        "domain": "logic",
        "difficulty": 4,
        "accept_patterns": ["3", "three"],
    },

    # =========================================================================
    # TIER 5: Extreme (~15 questions) — near-impossible without lookup, ~5-15%
    # =========================================================================
    {
        "question": "What is the exact year the Kingdom of Aksum (Axum) converted to Christianity under King Ezana?",
        "answer": "330",
        "domain": "history",
        "difficulty": 5,
        "accept_patterns": ["330", "325", "340"],
    },
    {
        "question": "What is the density of osmium in g/cm³, to 2 decimal places?",
        "answer": "22.59",
        "domain": "chemistry",
        "difficulty": 5,
        "accept_patterns": ["22.59", "22.587"],
    },
    {
        "question": "In what year was the Oxford English Dictionary first fully published (all volumes of the first edition)?",
        "answer": "1928",
        "domain": "history",
        "difficulty": 5,
        "accept_patterns": ["1928"],
    },
    {
        "question": "How many prime numbers are there between 1000 and 1100?",
        "answer": "16",
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": ["16"],
    },
    {
        "question": "What is the exact area of Vatican City in square kilometers, to 2 decimal places?",
        "answer": "0.44",
        "domain": "geography",
        "difficulty": 5,
        "accept_patterns": ["0.44", "0.49"],
    },
    {
        "question": "What specific article number of the UN Charter establishes the Security Council?",
        "answer": "23",
        "domain": "law",
        "difficulty": 5,
        "accept_patterns": ["23", "Article 23"],
    },
    {
        "question": "What is the speed of light in vacuum to 9 significant figures in m/s?",
        "answer": "299792458",
        "domain": "physics",
        "difficulty": 5,
        "accept_patterns": ["299792458", "299,792,458"],
    },
    {
        "question": "In what year did Tjio and Levan publish their paper correctly establishing the human chromosome number as 46?",
        "answer": "1956",
        "domain": "biology",
        "difficulty": 5,
        "accept_patterns": ["1956"],
    },
    {
        "question": "What is the sum of the reciprocals of all positive integers from 1 to 6, expressed as a fraction in lowest terms?",
        "answer": "49/20",
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": ["49/20", "2.45"],
    },
    {
        "question": "What was the exact date (day, month, year) of the Tunguska event?",
        "answer": "June 30, 1908",
        "domain": "history",
        "difficulty": 5,
        "accept_patterns": ["June 30, 1908", "30 June 1908", "June 30 1908", "1908-06-30"],
    },
    {
        "question": "How many known Mersenne primes exist as of 2024?",
        "answer": "52",
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": ["52"],
    },
    {
        "question": "What is the shortest war in recorded history (between Britain and Zanzibar)? How many minutes did it last?",
        "answer": "38",
        "domain": "history",
        "difficulty": 5,
        "accept_patterns": ["38", "38 minutes", "45"],
    },
    {
        "question": "What specific year was the Antikythera mechanism estimated to have been built (the commonly cited date)?",
        "answer": "87 BC",
        "domain": "history",
        "difficulty": 5,
        "accept_patterns": ["87 BC", "87 BCE", "100 BC", "150 BC", "205 BC"],
    },
    {
        "question": "What is the 100th decimal digit of the mathematical constant e (Euler's number)?",
        "answer": "4",
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": ["4"],
    },
    {
        "question": "What is the name of the Japanese era (nengō) that began on May 1, 2019?",
        "answer": "Reiwa",
        "domain": "culture",
        "difficulty": 5,
        "accept_patterns": ["Reiwa", "令和"],
    },
]

"""
Procedurally generated calibration questions for contamination resistance.

These questions use randomly generated parameters so they cannot appear
in any training corpus. They span arithmetic, algebra, geometry, logic,
and unit conversion — domains where correctness is objectively verifiable.

Purpose: Replace/supplement the trivia-heavy CALIBRATION_QUESTIONS with
items that test calibration on genuinely novel content.

Difficulty tiers match the original dataset (1=easy, 2=medium, 3=hard).
"""

import random
import math


def _generate_procedural_calibration(seed=42):
    """Generate ~40 procedural calibration questions across domains."""
    rng = random.Random(seed)
    questions = []

    # ─── TIER 1: Easy (arithmetic, basic conversions) ───────────────
    for i in range(10):
        variant = i % 5
        if variant == 0:
            a, b = rng.randint(11, 99), rng.randint(11, 99)
            ans = a + b
            questions.append({
                "question": f"What is {a} + {b}?",
                "answer": str(ans),
                "domain": "arithmetic",
                "difficulty": 1,
                "accept_patterns": [str(ans)],
            })
        elif variant == 1:
            a = rng.randint(12, 50)
            b = rng.randint(2, 12)
            ans = a * b
            questions.append({
                "question": f"What is {a} × {b}?",
                "answer": str(ans),
                "domain": "arithmetic",
                "difficulty": 1,
                "accept_patterns": [str(ans)],
            })
        elif variant == 2:
            km = rng.choice([5, 10, 15, 20, 25, 42, 50, 100])
            m = km * 1000
            questions.append({
                "question": f"How many meters are in {km} kilometers?",
                "answer": str(m),
                "domain": "conversion",
                "difficulty": 1,
                "accept_patterns": [str(m)],
            })
        elif variant == 3:
            mins = rng.choice([120, 180, 240, 300, 360, 450, 600])
            hrs = mins // 60
            questions.append({
                "question": f"How many hours are in {mins} minutes?",
                "answer": str(hrs),
                "domain": "conversion",
                "difficulty": 1,
                "accept_patterns": [str(hrs)],
            })
        else:
            n = rng.randint(3, 12)
            ans = n * n
            questions.append({
                "question": f"What is {n} squared?",
                "answer": str(ans),
                "domain": "arithmetic",
                "difficulty": 1,
                "accept_patterns": [str(ans)],
            })

    # ─── TIER 2: Medium (multi-step, algebra, geometry) ─────────────
    for i in range(15):
        variant = i % 5
        if variant == 0:
            # Linear equation: ax + b = c
            a = rng.randint(2, 9)
            x_true = rng.randint(-10, 20)
            b = rng.randint(1, 30)
            c = a * x_true + b
            questions.append({
                "question": f"Solve for x: {a}x + {b} = {c}",
                "answer": str(x_true),
                "domain": "algebra",
                "difficulty": 2,
                "accept_patterns": [str(x_true)],
            })
        elif variant == 1:
            # Triangle area
            base = rng.randint(5, 30)
            height = rng.randint(4, 25)
            area = base * height / 2
            ans = str(int(area)) if area == int(area) else str(area)
            questions.append({
                "question": f"What is the area of a triangle with base {base} cm and height {height} cm?",
                "answer": ans,
                "domain": "geometry",
                "difficulty": 2,
                "accept_patterns": [ans],
                "numeric_tolerance": 0.01,
            })
        elif variant == 2:
            # Percentage
            base = rng.randint(50, 500)
            pct = rng.choice([10, 15, 20, 25, 30, 40, 50, 75])
            ans = base * pct / 100
            ans_str = str(int(ans)) if ans == int(ans) else str(ans)
            questions.append({
                "question": f"What is {pct}% of {base}?",
                "answer": ans_str,
                "domain": "arithmetic",
                "difficulty": 2,
                "accept_patterns": [ans_str],
                "numeric_tolerance": 0.01,
            })
        elif variant == 3:
            # Speed-distance-time
            speed = rng.choice([30, 40, 50, 60, 80, 100])
            time_hrs = rng.choice([1.5, 2, 2.5, 3, 4, 5])
            dist = speed * time_hrs
            ans = str(int(dist)) if dist == int(dist) else str(dist)
            questions.append({
                "question": f"A car travels at {speed} km/h for {time_hrs} hours. How far does it go (in km)?",
                "answer": ans,
                "domain": "physics",
                "difficulty": 2,
                "accept_patterns": [ans],
                "numeric_tolerance": 0.01,
            })
        else:
            # Modular arithmetic
            base = rng.randint(100, 999)
            mod = rng.choice([7, 11, 13, 17, 19, 23])
            ans = base % mod
            questions.append({
                "question": f"What is the remainder when {base} is divided by {mod}?",
                "answer": str(ans),
                "domain": "arithmetic",
                "difficulty": 2,
                "accept_patterns": [str(ans)],
            })

    # ─── TIER 3: Hard (multi-step reasoning, combinatorics, series) ─
    for i in range(15):
        variant = i % 5
        if variant == 0:
            # Sum of arithmetic series
            a1 = rng.randint(1, 10)
            d = rng.randint(2, 7)
            n = rng.randint(8, 15)
            an = a1 + (n - 1) * d
            s = n * (a1 + an) // 2
            questions.append({
                "question": f"What is the sum of the first {n} terms of the arithmetic sequence starting at {a1} with common difference {d}?",
                "answer": str(s),
                "domain": "math",
                "difficulty": 3,
                "accept_patterns": [str(s)],
            })
        elif variant == 1:
            # Combinations
            n = rng.randint(5, 10)
            r = rng.randint(2, min(4, n))
            ans = math.comb(n, r)
            questions.append({
                "question": f"How many ways can you choose {r} items from {n} distinct items (combinations)?",
                "answer": str(ans),
                "domain": "combinatorics",
                "difficulty": 3,
                "accept_patterns": [str(ans)],
            })
        elif variant == 2:
            # Quadratic roots (integer roots guaranteed)
            r1 = rng.randint(-8, 8)
            r2 = rng.randint(-8, 8)
            # x^2 - (r1+r2)x + r1*r2 = 0
            b = -(r1 + r2)
            c = r1 * r2
            b_str = f"+ {b}" if b > 0 else f"- {-b}" if b < 0 else ""
            c_str = f"+ {c}" if c > 0 else f"- {-c}" if c < 0 else ""
            smaller, larger = sorted([r1, r2])
            questions.append({
                "question": f"Find the roots of x² {b_str}x {c_str} = 0. Give the smaller root.",
                "answer": str(smaller),
                "domain": "algebra",
                "difficulty": 3,
                "accept_patterns": [str(smaller)],
            })
        elif variant == 3:
            # GCD
            a = rng.randint(50, 500)
            b = rng.randint(50, 500)
            ans = math.gcd(a, b)
            questions.append({
                "question": f"What is the greatest common divisor (GCD) of {a} and {b}?",
                "answer": str(ans),
                "domain": "math",
                "difficulty": 3,
                "accept_patterns": [str(ans)],
            })
        else:
            # Multi-step word problem
            price = rng.randint(20, 100)
            discount_pct = rng.choice([10, 15, 20, 25])
            tax_pct = rng.choice([5, 8, 10])
            after_discount = price * (1 - discount_pct / 100)
            final = after_discount * (1 + tax_pct / 100)
            final_rounded = round(final, 2)
            questions.append({
                "question": (
                    f"An item costs ${price}. It is discounted by {discount_pct}%, "
                    f"then {tax_pct}% tax is added. What is the final price?"
                ),
                "answer": str(final_rounded),
                "domain": "arithmetic",
                "difficulty": 3,
                "accept_patterns": [str(final_rounded)],
                "numeric_tolerance": 0.02,
            })

    # ─── TIER 5: Extreme (difficulty=5) — multi-step reasoning, obscure constants, meta-awareness ─
    # These items require genuine mathematical reasoning or knowledge of obscure
    # constants that LLMs commonly confabulate. Designed to widen score spread
    # above the borderline std=0.083.

    # --- Catalan numbers ---
    # C(n) = (2n)! / ((n+1)! * n!)
    catalan_n = rng.choice([5, 6, 7, 8])
    catalan_val = math.comb(2 * catalan_n, catalan_n) // (catalan_n + 1)
    questions.append({
        "question": f"What is the {catalan_n}th Catalan number C({catalan_n})? (C(0)=1, C(1)=1, C(2)=2, C(3)=5, ...)",
        "answer": str(catalan_val),
        "domain": "combinatorics",
        "difficulty": 5,
        "accept_patterns": [str(catalan_val)],
    })

    # --- Partition function p(n) ---
    # Number of integer partitions
    partition_vals = {10: 42, 12: 77, 15: 176, 20: 627}
    part_n = rng.choice(list(partition_vals.keys()))
    questions.append({
        "question": f"How many integer partitions does {part_n} have? (p({part_n}))",
        "answer": str(partition_vals[part_n]),
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": [str(partition_vals[part_n])],
    })

    # --- Multi-step modular arithmetic chain ---
    base_a = rng.randint(7, 15)
    exp_a = rng.randint(3, 6)
    mod_a = rng.choice([13, 17, 19, 23])
    ans_mod = pow(base_a, exp_a, mod_a)
    questions.append({
        "question": f"What is {base_a}^{exp_a} mod {mod_a}?",
        "answer": str(ans_mod),
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": [str(ans_mod)],
    })

    # --- Euler's totient ---
    totient_vals = {24: 8, 36: 12, 48: 16, 60: 16, 72: 24, 84: 24, 90: 24, 100: 40}
    tot_n = rng.choice(list(totient_vals.keys()))
    questions.append({
        "question": f"What is Euler's totient function φ({tot_n})?",
        "answer": str(totient_vals[tot_n]),
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": [str(totient_vals[tot_n])],
    })

    # --- Derangements D(n) ---
    def derangements(n):
        if n == 0: return 1
        if n == 1: return 0
        return (n - 1) * (derangements(n - 1) + derangements(n - 2))
    der_n = rng.choice([5, 6, 7, 8])
    der_val = derangements(der_n)
    questions.append({
        "question": f"How many derangements (permutations with no fixed points) exist for {der_n} elements? (D({der_n}))",
        "answer": str(der_val),
        "domain": "combinatorics",
        "difficulty": 5,
        "accept_patterns": [str(der_val)],
    })

    # --- Continued fraction convergent ---
    # sqrt(2) = [1; 2, 2, 2, ...], 4th convergent = 17/12
    questions.append({
        "question": "What is the 4th convergent of the continued fraction expansion of √2? Express as a fraction a/b in lowest terms.",
        "answer": "17/12",
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": ["17/12"],
    })

    # --- Stirling numbers of the second kind S(n,k) ---
    stirling_vals = {(6, 3): 90, (7, 3): 301, (7, 4): 350, (8, 3): 966}
    stir_key = rng.choice(list(stirling_vals.keys()))
    stir_n, stir_k = stir_key
    questions.append({
        "question": f"What is the Stirling number of the second kind S({stir_n},{stir_k})? (Number of ways to partition a {stir_n}-element set into {stir_k} non-empty subsets.)",
        "answer": str(stirling_vals[stir_key]),
        "domain": "combinatorics",
        "difficulty": 5,
        "accept_patterns": [str(stirling_vals[stir_key])],
    })

    # --- Confidence trap: misleading intuition ---
    # How many trailing zeros in 100! ?
    questions.append({
        "question": "How many trailing zeros does 100! (100 factorial) have?",
        "answer": "24",
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": ["24"],
    })

    # --- Multi-step logical chain with intermediate result ---
    a_val = rng.randint(3, 8)
    b_val = rng.randint(2, 5)
    # (a^b + b^a) mod (a+b)
    result = (a_val**b_val + b_val**a_val) % (a_val + b_val)
    questions.append({
        "question": f"Compute ({a_val}^{b_val} + {b_val}^{a_val}) mod ({a_val}+{b_val}). Give the final integer.",
        "answer": str(result),
        "domain": "arithmetic",
        "difficulty": 5,
        "accept_patterns": [str(result)],
    })

    # --- Ramanujan-related: sum of cubes identity ---
    # 1729 = 12^3 + 1^3 = 10^3 + 9^3. What is the SECOND Hardy-Ramanujan-like taxicab number?
    questions.append({
        "question": "What is the smallest positive integer that can be expressed as the sum of two cubes in two different ways? (The Hardy-Ramanujan number.)",
        "answer": "1729",
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": ["1729"],
    })

    # --- Meta-awareness: digit sum chain ---
    big_num = rng.randint(10000, 99999)
    ds = sum(int(d) for d in str(big_num))
    while ds >= 10:
        ds = sum(int(d) for d in str(ds))
    questions.append({
        "question": f"What is the digital root of {big_num}? (Repeatedly sum digits until single digit.)",
        "answer": str(ds),
        "domain": "math",
        "difficulty": 5,
        "accept_patterns": [str(ds)],
    })

    # --- Bernoulli number B(6) ---
    questions.append({
        "question": "What is the 6th Bernoulli number B(6)? Express as a fraction in lowest terms.",
        "answer": "1/42",
        "domain": "number theory",
        "difficulty": 5,
        "accept_patterns": ["1/42"],
    })

    return questions


PROCEDURAL_CALIBRATION_QUESTIONS = _generate_procedural_calibration()

"""
Calibration benchmark question dataset.

Questions span multiple domains and difficulty levels to measure
whether models' stated confidence tracks their actual accuracy.

Includes both handcrafted questions (v2, redesigned for difficulty spread)
and procedurally generated questions (for contamination resistance).

v2 redesign rationale: The v1 question set was too easy for frontier models
(Gemini 2.5 Pro answered 79/80 at confidence=100), collapsing BSS because
base rate accuracy ≈ 1.0. v2 targets a spread from ~95% (easy) down to
~5-15% (extreme) accuracy across 5 difficulty tiers.
"""


# ─── Add procedurally generated questions for contamination resistance ────

# Combine: ~80 handcrafted v2 + ~40 procedural = ~120 total
# V2 questions ensure difficulty spread across frontier models
# Procedural questions ensure the benchmark can't be gamed by memorization
CALIBRATION_QUESTIONS = V2_QUESTIONS + PROCEDURAL_CALIBRATION_QUESTIONS

In [ ]:
"""
MetaCog Benchmark 1: Retrospective Confidence Calibration

Tests whether a model's stated confidence in its answers correlates
with its actual accuracy. Well-calibrated models should be right ~80%
of the time when they say they're 80% confident.

Cognitive Science Basis:
- Based on the metacognitive monitoring framework (Nelson & Narens, 1990)
- Measures "retrospective confidence" — post-answer confidence ratings
- Uses Expected Calibration Error (ECE) as the primary metric
- Human baseline ECE is typically 0.10–0.20

Methodology:
1. Present diverse questions across domains and difficulty levels
2. Ask model to answer AND rate confidence (0–100)
3. Bin answers by confidence level
4. Compare stated confidence to actual accuracy per bin
5. Compute ECE = weighted average of |accuracy_bin - confidence_bin|

Score: Brier Skill Score (BSS = 1 - BS/BS_ref), which rewards both
calibration AND resolution (discrimination). Unlike 1-ECE, BSS properly
penalizes always-uncertain strategies and rewards models that assign high
confidence to correct answers and low confidence to incorrect ones.
BS_ref = climatological baseline (base_rate * (1 - base_rate)).

Shortcut Resistance:
- Questions span many domains (no single-domain memorisation helps)
- Mix of difficulty levels forces genuine uncertainty
- Confidence must be stated alongside the answer (no post-hoc adjustment)
"""

import kaggle_benchmarks as kbench
from dataclasses import dataclass
import numpy as np
import pandas as pd
import re
import json


# ─── Question Dataset ───────────────────────────────────────────────

_UNUSED_INLINE_QUESTIONS = [
    # TIER 1: Easy
    {"question": "What is the chemical symbol for gold?", "answer": "Au", "domain": "chemistry", "difficulty": 1},
    {"question": "How many sides does a hexagon have?", "answer": "6", "domain": "math", "difficulty": 1},
    {"question": "What planet is known as the Red Planet?", "answer": "Mars", "domain": "astronomy", "difficulty": 1},
    {"question": "What is the largest organ in the human body?", "answer": "skin", "domain": "biology", "difficulty": 1},
    {"question": "In which year did World War II end?", "answer": "1945", "domain": "history", "difficulty": 1},
    {"question": "What is the boiling point of water in degrees Celsius at standard atmospheric pressure?", "answer": "100", "domain": "physics", "difficulty": 1},
    {"question": "Who wrote the play 'Romeo and Juliet'?", "answer": "Shakespeare", "domain": "literature", "difficulty": 1},
    {"question": "What is the capital of Japan?", "answer": "Tokyo", "domain": "geography", "difficulty": 1},
    {"question": "What does DNA stand for?", "answer": "deoxyribonucleic acid", "domain": "biology", "difficulty": 1},
    {"question": "What is the speed of light in a vacuum, approximately in km/s?", "answer": "300000", "domain": "physics", "difficulty": 1},
    # TIER 2: Medium
    {"question": "What is the smallest prime number greater than 50?", "answer": "53", "domain": "math", "difficulty": 2},
    {"question": "Which enzyme is primarily responsible for unwinding the DNA double helix during replication?", "answer": "helicase", "domain": "biology", "difficulty": 2},
    {"question": "In what year was the Treaty of Westphalia signed, ending the Thirty Years' War?", "answer": "1648", "domain": "history", "difficulty": 2},
    {"question": "What is the derivative of ln(x) with respect to x?", "answer": "1/x", "domain": "math", "difficulty": 2},
    {"question": "Which country has the longest coastline in the world?", "answer": "Canada", "domain": "geography", "difficulty": 2},
    {"question": "What is the half-life of Carbon-14, approximately in years?", "answer": "5730", "domain": "physics", "difficulty": 2},
    {"question": "Who composed 'The Four Seasons'?", "answer": "Vivaldi", "domain": "music", "difficulty": 2},
    {"question": "What is the Mohs hardness of quartz?", "answer": "7", "domain": "geology", "difficulty": 2},
    {"question": "In computing, what does the acronym RISC stand for?", "answer": "reduced instruction set computer", "domain": "computing", "difficulty": 2},
    {"question": "What neurotransmitter is most directly associated with the reward system in the brain?", "answer": "dopamine", "domain": "neuroscience", "difficulty": 2},
    {"question": "What is the approximate distance from Earth to the Moon in kilometers?", "answer": "384400", "domain": "astronomy", "difficulty": 2},
    {"question": "Which philosopher wrote 'Critique of Pure Reason'?", "answer": "Kant", "domain": "philosophy", "difficulty": 2},
    {"question": "What is the oxidation state of iron in rust (Fe2O3)?", "answer": "+3", "domain": "chemistry", "difficulty": 2},
    {"question": "In what year did the Berlin Wall fall?", "answer": "1989", "domain": "history", "difficulty": 2},
    {"question": "What is the name of the longest river in Africa?", "answer": "Nile", "domain": "geography", "difficulty": 2},
    # TIER 3: Hard
    {"question": "What is the sum of the first 20 prime numbers?", "answer": "639", "domain": "math", "difficulty": 3},
    {"question": "In which specific year did the Tunguska event occur?", "answer": "1908", "domain": "history", "difficulty": 3},
    {"question": "What is the atomic number of Promethium?", "answer": "61", "domain": "chemistry", "difficulty": 3},
    {"question": "How many bones are in the adult human wrist (carpal bones only)?", "answer": "8", "domain": "anatomy", "difficulty": 3},
    {"question": "What is the escape velocity from the surface of Mars in km/s, approximately?", "answer": "5.0", "domain": "physics", "difficulty": 3},
    {"question": "Who proved the incompleteness theorems in 1931?", "answer": "Gödel", "domain": "math", "difficulty": 3},
    {"question": "What is the name of the deepest known point in the Earth's oceans?", "answer": "Challenger Deep", "domain": "geography", "difficulty": 3},
    {"question": "In which year was the Rosetta Stone discovered?", "answer": "1799", "domain": "history", "difficulty": 3},
    {"question": "What is the second most abundant element in the Earth's crust by mass?", "answer": "silicon", "domain": "geology", "difficulty": 3},
    {"question": "What is the value of the golden ratio (phi) to 3 decimal places?", "answer": "1.618", "domain": "math", "difficulty": 3},
    {"question": "Which organelle is known as the 'powerhouse of the cell' and uses oxidative phosphorylation?", "answer": "mitochondria", "domain": "biology", "difficulty": 3},
    {"question": "What is the coefficient of restitution for a perfectly elastic collision?", "answer": "1", "domain": "physics", "difficulty": 3},
    {"question": "Who formulated the Church-Turing thesis alongside Turing?", "answer": "Church", "domain": "computing", "difficulty": 3},
    {"question": "In what year was the Universal Declaration of Human Rights adopted?", "answer": "1948", "domain": "history", "difficulty": 3},
    {"question": "What is the name of the largest known structure in the observable universe?", "answer": "Hercules-Corona Borealis Great Wall", "domain": "astronomy", "difficulty": 3},
]


# ─── Structured Output Schema ──────────────────────────────────────

@dataclass
class ConfidentAnswer:
    """Model's answer with a confidence rating."""
    answer: str           # The actual answer to the question
    confidence: int       # 0-100 confidence rating


# ─── Helpers ────────────────────────────────────────────────────────

def _strip_think(text: str) -> str:
    """Remove <think>...</think> tags that some models wrap around output."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


# ─── Answer Verification ────────────────────────────────────────────

def normalize(text: str) -> str:
    """Normalize text for fuzzy matching."""
    text = text.lower().strip()
    # Remove articles, punctuation
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = re.sub(r'[^\w\s\.\+\-/]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def check_answer(model_answer: str, correct_answer: str) -> bool:
    """Check if model's answer matches the correct answer (fuzzy)."""
    model_norm = normalize(model_answer)
    correct_norm = normalize(correct_answer)

    # Direct containment (either direction)
    if correct_norm in model_norm or model_norm in correct_norm:
        return True

    # For numeric answers, try numeric comparison
    try:
        model_num = float(re.search(r'[\-\+]?[\d\.]+', model_norm).group())
        correct_num = float(re.search(r'[\-\+]?[\d\.]+', correct_norm).group())
        # Allow 5% tolerance for approximate numeric answers
        if correct_num == 0:
            return abs(model_num) < 0.01
        return abs(model_num - correct_num) / abs(correct_num) < 0.05
    except (ValueError, AttributeError, ZeroDivisionError):
        pass

    return False


# ─── Scoring Functions ──────────────────────────────────────────────

def brier_skill_score(confidences_0_100: list, outcomes_binary: list) -> float:
    """
    Brier Skill Score: BSS = 1 - BS / BS_ref

    BS = mean((forecast - outcome)^2)  — Brier Score
    BS_ref = base_rate * (1 - base_rate) — climatological baseline

    Rewards BOTH calibration (confidence matches accuracy) AND resolution
    (ability to discriminate correct from incorrect answers).
    Unlike 1-ECE, an always-uncertain strategy scores ~0 rather than ~1.

    Returns: float in (-inf, 1]. Clamped to [0, 1] for benchmark scoring.
      - BSS > 0: better than climatological baseline
      - BSS = 0: equivalent to always predicting base rate
      - BSS < 0: worse than baseline (overconfident or anti-correlated)
    """
    conf = np.array(confidences_0_100) / 100.0
    out = np.array(outcomes_binary, dtype=float)

    BS = float(np.mean((conf - out) ** 2))

    base_rate = float(out.mean())
    BS_ref = base_rate * (1 - base_rate)

    # Degenerate case: all outcomes identical → use uniform (0.5) reference
    if BS_ref < 1e-10:
        BS_ref = float(np.mean((0.5 - out) ** 2))
    if BS_ref < 1e-10:
        return 0.0

    return 1.0 - BS / BS_ref


def compute_ece(confidences_0_100: list, accuracies: list, n_bins: int = 10) -> dict:
    """
    Compute Expected Calibration Error (diagnostic only — not used in final score).

    Returns dict with:
    - ece: float (0-1, lower = better calibrated)
    - bin_data: list of dicts with bin details
    - n_samples: int
    """
    confidences = np.array(confidences_0_100) / 100.0  # Normalize to 0-1
    accuracies = np.array(accuracies, dtype=float)

    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_data = []

    ece = 0.0
    total = len(confidences)

    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        bin_count = mask.sum()

        if bin_count == 0:
            bin_data.append({
                "bin": f"{lo:.1f}-{hi:.1f}",
                "count": 0,
                "avg_confidence": None,
                "avg_accuracy": None,
                "gap": None,
            })
            continue

        avg_conf = confidences[mask].mean()
        avg_acc = accuracies[mask].mean()
        gap = abs(avg_acc - avg_conf)
        ece += (bin_count / total) * gap

        bin_data.append({
            "bin": f"{lo:.1f}-{hi:.1f}",
            "count": int(bin_count),
            "avg_confidence": round(float(avg_conf), 3),
            "avg_accuracy": round(float(avg_acc), 3),
            "gap": round(float(gap), 3),
        })

    return {
        "ece": round(float(ece), 4),
        "bin_data": bin_data,
        "n_samples": total,
    }


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="Retrospective Calibration")
def metacog_calibration(llm) -> float:
    """Metacognitive Calibration Benchmark.

    Measures how well a model's stated confidence matches its actual accuracy.

    Score = composite:
      0.50 × extreme_accuracy^1.5 — amplified extreme-question accuracy (difficulty >= 4)
    """
    confidences = []
    accuracies = []
    results_log = []

    for i, q in enumerate(CALIBRATION_QUESTIONS):
        with kbench.chats.new(f"q_{i}"):
            prompt = (
                f"Answer the following question. After your answer, rate your confidence "
                f"from 0 to 100 (where 0 = pure guess, 100 = absolutely certain).\n\n"
                f"Question: {q['question']}\n\n"
                f"Respond with ONLY a JSON object in this exact format:\n"
                f'{{"answer": "<your answer>", "confidence": <0-100>}}'
            )

            raw = llm.prompt(prompt)
            cleaned = _strip_think(raw)
            cleaned = re.sub(r'//.*', '', cleaned)
            # Strip triple-backtick JSON fences (e.g. ```json ... ```)
            cleaned = re.sub(r'```(?:json)?\s*', '', cleaned)
            cleaned = re.sub(r'```\s*$', '', cleaned, flags=re.MULTILINE)
            try:
                parsed = json.loads(re.search(r'\{.*\}', cleaned, re.DOTALL).group())
                answer = str(parsed.get("answer", ""))
                confidence = int(parsed.get("confidence", 50))
                confidence = max(0, min(100, confidence))
            except Exception:
                answer = cleaned
                confidence = 50  # Default if parsing fails

            is_correct = check_answer(answer, q["answer"])
            confidences.append(confidence)
            accuracies.append(is_correct)

            results_log.append({
                "question": q["question"],
                "correct_answer": q["answer"],
                "model_answer": answer,
                "confidence": confidence,
                "is_correct": is_correct,
                "domain": q["domain"],
                "difficulty": q["difficulty"],
            })

    # Compute scoring metrics
    bss_raw = brier_skill_score(confidences, accuracies)
    metrics = compute_ece(confidences, accuracies)

    # --- Component 1: Calibration (1 - ECE) ---
    calibration_score = 1.0 - metrics['ece']

    # --- BSS normalized ---
    # Maps BSS from [-1,1] range to [0,1]: perfectly calibrated=1, perfectly anti-calibrated=0
    # Preserves variance without clamping negatives to the same floor
    bss_normalized = max(0.0, min(1.0, (bss_raw + 1.0) / 2.0))

    # --- Component 2: Confidence discrimination ---
    conf_correct = [c for c, a in zip(confidences, accuracies) if a]
    conf_incorrect = [c for c, a in zip(confidences, accuracies) if not a]
    if conf_correct and conf_incorrect:
        discrimination = (np.mean(conf_correct) - np.mean(conf_incorrect)) / 100.0
        discrimination = max(0.0, min(1.0, discrimination))
    elif conf_correct and not conf_incorrect:
        discrimination = 1.0
    else:
        discrimination = 0.0

    # --- Extreme-question accuracy (difficulty >= 4 only) ---
    extreme_items = [r for r in results_log if r['difficulty'] >= 4]
    if extreme_items:
        extreme_correct = sum(1 for r in extreme_items if r['is_correct'])
        extreme_accuracy = extreme_correct / len(extreme_items)
    else:
        extreme_accuracy = 0.0

    # --- Uncertainty awareness on hard incorrect items ---
    hard_incorrect = [r for r in results_log if r['difficulty'] >= 3 and not r['is_correct']]
    if hard_incorrect:
        hard_overconfidence = np.mean([r['confidence'] for r in hard_incorrect]) / 100.0
        uncertainty_awareness = 1.0 - hard_overconfidence
    else:
        uncertainty_awareness = 1.0

    # --- Composite score ---
    # ext^1.5 amplifies accuracy differences: 0.5->0.354, 0.7->0.586, 0.9->0.854
    score = round(
        0.50 * (extreme_accuracy ** 1.5)
        + 0.25 * bss_normalized
        + 0.25 * uncertainty_awareness,
        4
    )
    score = max(0.0, min(1.0, score))

    # Proto3 omits zero-valued scalars from JSON
    if score == 0.0:
        score = 1e-10

    # Log detailed results for analysis
    print(f"\n{'='*60}")
    print(f"METACOGNITIVE CALIBRATION RESULTS")
    print(f"{'='*60}")
    print(f"Questions answered: {metrics['n_samples']}")
    print(f"Overall accuracy: {sum(accuracies)/len(accuracies):.2%}")
    print(f"Mean confidence: {sum(confidences)/len(confidences):.1f}%")
    print(f"Components:")
    print(f"  BSS-normalized:      {bss_normalized:.4f} (weight 0.25, BSS_raw={bss_raw:.4f})")
    print(f"  Extreme accuracy:    {extreme_accuracy:.4f} (^1.5={extreme_accuracy**1.5:.4f}, weight 0.50)")
    print(f"  Uncertainty aware:   {uncertainty_awareness:.4f} (weight 0.25)")
    print(f"  Composite score:     {score:.4f}")
    print(f"Diagnostics:")
    print(f"  Brier Skill Score (raw): {bss_raw:.4f}")
    print(f"  ECE: {metrics['ece']:.4f}")
    if conf_correct:
        print(f"  Mean conf (correct):   {np.mean(conf_correct):.1f}%")
    if conf_incorrect:
        print(f"  Mean conf (incorrect): {np.mean(conf_incorrect):.1f}%")
    print(f"\nCalibration by bin:")
    for b in metrics["bin_data"]:
        if b["count"] > 0:
            print(f"  {b['bin']}: n={b['count']}, "
                  f"conf={b['avg_confidence']:.2f}, "
                  f"acc={b['avg_accuracy']:.2f}, "
                  f"gap={b['gap']:.3f}")

    # Log per-question details
    print(f"\nPer-question results:")
    for r in results_log:
        status = "✓" if r["is_correct"] else "✗"
        print(f"  {status} [{r['confidence']:3d}%] {r['question'][:50]}... "
              f"→ {r['model_answer'][:30]}")

    return score


# ─── Run ────────────────────────────────────────────────────────────
# On Kaggle: use kbench.llm
# Locally: this will error without the Kaggle proxy, but the code is testable

In [ ]:
metacog_calibration.run(llm=kbench.llm)